In this notebook we use the network learned on images cropped in the bounding boxes to generate a new dataset. From the original data set `Imagenet_full` we will fixate on the most likely position and generate the `Imagenet_focus` dataset

TODO: use rotations of the image to find the center of the object


In [ ]:
import retinoto_py as fovea
args = fovea.Params(do_fovea=True, batch_size=1, shuffle=False)
print(args)

In [ ]:
resolution = (21, 34) # landscape images
# resolution = (21, 21)
# resolution = (13, 21) # landscape images
N_fixations = fovea.np.prod(resolution)


size_ratios = [0.4, 0.4, 0.6, 0.4, 0.4, 0.9]
# angles=[0, 60, 120]
angles=[0, 90, 180]


format = 'png'
subset_factor = 1


## Compute position and sizes


In [ ]:
from torchvision.io import read_image

FULL_DATA_DIR = args.DATAROOT / 'Imagenet_full'
FOCUS_DATA_DIR = args.DATAROOT / 'Imagenet_focus'
FOCUS_DATA_DIR.mkdir(exist_ok=True)

IMG_EXTS = {'.jpg', '.jpeg', '.JPEG', '.png', '.bmp'}
def clean_list(list_dir, EXCLUDED_FILES={'.DS_Store', '.ipynb_checkpoints'}):
    return [ p for p in list_dir if p.is_file() and p.name not in EXCLUDED_FILES  ]
# parameters for the new dataset


from torchvision.transforms.functional import InterpolationMode, resize

args = fovea.Params(do_fovea=True, model_name='convnext_base', batch_size=1, shuffle=False, subset_factor=subset_factor)
model = fovea.load_model(args, model_filename=args.data_cache / f'32_fovea_model_name=convnext_base_dataset=bbox.pth')

for folder in ['val', 'train']:
    print(f'\n Scanning folder "{folder}"')
    DATA_DIR = args.DATAROOT / 'Imagenet_full' / folder
    dataset = fovea.get_dataset(args, DATA_DIR, do_full_preprocess=False)
    loader = fovea.get_loader(args, dataset)
    class_to_idx = dataset.class_to_idx


    src_root = FULL_DATA_DIR / folder
    tgt_root = FOCUS_DATA_DIR / folder
    tgt_root.mkdir(parents=True, exist_ok=True)

    count_in = 0
    count_out = 0

    # parcours récursif avec pathlib
    for img_path in fovea.tqdm(clean_list(list(src_root.rglob('*.*')))):
        if not img_path.is_file() or img_path.suffix not in IMG_EXTS:
            print(f'File {img_path} is detected as an invalid image.')
            continue

        count_in += 1
        imgid = img_path.stem

        class_id = img_path.parent.name
        true_idx = class_to_idx[class_id]
        target_folder = tgt_root / class_id
        target_folder.mkdir(parents=True, exist_ok=True)
        out_path = target_folder / f'{imgid}.{format}'

        try:
            image = read_image(img_path)/255.
            if image.shape[0] == 1: image = image.repeat(3, 1, 1)
            three, H, W = image.shape
            assert three == 3

        except Exception as e:
            print(f' could not open {img_path}: {e}')
            break


        image = image.squeeze(0)
        three, H, W = image.shape
        pos_H, pos_W = fovea.get_positions(H, W, resolution=resolution)
        probas = fovea.compute_likelihood_map(args, model, image, pos_H, pos_W, size_ratio=size_ratio)    
        likelihood_map = probas[:, true_idx]

        if sigma > 0: likelihood_map = gaussian_filter(likelihood_map, sigma=sigma)

        likelihood_max, idx_pos = likelihood_map.max(axis=-1)

        box_size = int( fovea.np.sqrt(H*W)*size_ratio)
        image_fix = fovea.fixate(image, int(pos_H[idx_pos]), int(pos_W[idx_pos]), box_size) 

        img_pil = fovea.TF.to_pil_image(image_fix)
        img_pil.save(out_path, format=format)

        count_out += 1

    print(f' - in: {count_in} / out: {count_out}')

## Building the new dataset


In [ ]:
# from torchvision.io import read_image

# FULL_DATA_DIR = args.DATAROOT / 'Imagenet_full'
# FOCUS_DATA_DIR = args.DATAROOT / 'Imagenet_focus'
# FOCUS_DATA_DIR.mkdir(exist_ok=True)

# IMG_EXTS = {'.jpg', '.jpeg', '.JPEG', '.png', '.bmp'}
# def clean_list(list_dir, EXCLUDED_FILES={'.DS_Store', '.ipynb_checkpoints'}):
#     return [ p for p in list_dir if p.is_file() and p.name not in EXCLUDED_FILES  ]
# # parameters for the new dataset


# from torchvision.transforms.functional import InterpolationMode, resize

# args = fovea.Params(do_fovea=True, model_name='convnext_base', batch_size=1, shuffle=False, subset_factor=subset_factor)
# model = fovea.load_model(args, model_filename=args.data_cache / f'32_fovea_model_name=convnext_base_dataset=bbox.pth')

# for folder in ['val', 'train']:
#     print(f'\n Scanning folder "{folder}"')
#     DATA_DIR = args.DATAROOT / 'Imagenet_full' / folder
#     dataset = fovea.get_dataset(args, DATA_DIR, do_full_preprocess=False)
#     loader = fovea.get_loader(args, dataset)
#     class_to_idx = dataset.class_to_idx


#     src_root = FULL_DATA_DIR / folder
#     tgt_root = FOCUS_DATA_DIR / folder
#     tgt_root.mkdir(parents=True, exist_ok=True)

#     count_in = 0
#     count_out = 0

#     # parcours récursif avec pathlib
#     for img_path in fovea.tqdm(clean_list(list(src_root.rglob('*.*')))):
#         if not img_path.is_file() or img_path.suffix not in IMG_EXTS:
#             print(f'File {img_path} is detected as an invalid image.')
#             continue

#         count_in += 1
#         imgid = img_path.stem

#         class_id = img_path.parent.name
#         true_idx = class_to_idx[class_id]
#         target_folder = tgt_root / class_id
#         target_folder.mkdir(parents=True, exist_ok=True)
#         out_path = target_folder / f'{imgid}.{format}'

#         try:
#             image = read_image(img_path)/255.
#             if image.shape[0] == 1: image = image.repeat(3, 1, 1)
#             three, H, W = image.shape
#             assert three == 3

#         except Exception as e:
#             print(f' could not open {img_path}: {e}')
#             break


#         image = image.squeeze(0)
#         three, H, W = image.shape
#         pos_H, pos_W = fovea.get_positions(H, W, resolution=resolution)
#         probas = fovea.compute_likelihood_map(args, model, image, pos_H, pos_W, size_ratio=size_ratio)    
#         likelihood_map = probas[:, true_idx]

#         if sigma > 0: likelihood_map = gaussian_filter(likelihood_map, sigma=sigma)

#         likelihood_max, idx_pos = likelihood_map.max(axis=-1)

#         box_size = int( fovea.np.sqrt(H*W)*size_ratio)
#         image_fix = fovea.fixate(image, int(pos_H[idx_pos]), int(pos_W[idx_pos]), box_size) 

#         img_pil = fovea.TF.to_pil_image(image_fix)
#         img_pil.save(out_path, format=format)

#         count_out += 1

#     print(f' - in: {count_in} / out: {count_out}')

Voilà !